In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from sklearn.model_selection import train_test_split
import os

# ── 参数配置 ──────────────────────────────────────────
INPUT_PATH   = "20260127_passivate_merged_clean.csv"          # 原始数据路径
OUTPUT_DIR   = "finetune_data/perovskite"    # 输出目录
VAL_RATIO    = 0.1
TEST_RATIO   = 0.1
SEED         = 42
# ──────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(INPUT_PATH)
print(f"原始数据: {len(df)} 行")
df.head(3)

原始数据: 249 行


,Unnamed: 0,row_id,case_id,data_item,title,doi,pdf_path,judge_judgment,reviewer_judgment,final_judgment,...,control_pce,relative_pce,control_type,stability_test_type,stability_duration_hours,stability_retention_pct,mechanism_defect_healing,mechanism_structural,mechanism_barrier,mechanism_energetic
0,0,184,109,3,Bandgap-universal passivation enables stable p...,10.1126/science.ado2302,NaN,relevant,relevant,relevant,...,20.0,1.040000,Reference,NaN,0.0,0.0,True,False,False,False
1,1,39,27,1,Evolution of defects during the degradation of...,NaN,NaN,relevant,relevant,relevant,...,18.0,0.976111,NaN,NaN,0.0,0.0,True,False,True,True
2,2,175,105,1,Bi-functional interfaces by poly(ionic liquid)...,NaN,NaN,relevant,relevant,relevant,...,19.0,1.126316,Untreated,Maximum power point tracking under 1-sun illum...,700.0,80.0,True,False,True,True


In [2]:
df_clean = df.copy()

# 1. 计算 y = pce - control_pce
df_clean['y'] = pd.to_numeric(df_clean['pce'], errors='coerce') \
              - pd.to_numeric(df_clean['control_pce'], errors='coerce')

# 2. 重命名 SMILES1 -> smiles，清理无效值
df_clean = df_clean.rename(columns={'SMILES1': 'smiles'})
df_clean['smiles'] = df_clean['smiles'].astype(str).str.strip()
df_clean = df_clean[
    df_clean['smiles'].notna() &
    (df_clean['smiles'] != '') &
    (df_clean['smiles'].str.lower() != 'nan')
]

# 3. 去除 y 为 NaN 的行
df_clean = df_clean.dropna(subset=['y'])

# 4. 只保留 smiles 和 y
df_clean = df_clean[['smiles', 'y']].reset_index(drop=True)

print(f"清理后数据: {len(df_clean)} 行")
print(f"y 统计:\n{df_clean['y'].describe()}")
df_clean.head()

清理后数据: 249 行
y 统计:
count    249.000000
mean       1.775341
std        1.709189
min       -7.730000
25%        1.140000
50%        1.890000
75%        2.600000
max        6.810000
Name: y, dtype: float64


,smiles,y
0,CO[Si](CCCNCCNCCN)(OC)OC,0.80
1,CCCCCCCC[NH3+].CCCCCCCC[NH3+].[O-]S(=O)(=O)[O-],-0.43
2,CN1C=C[N+](C)=C1.FC(F)(F)S(=O)(=O)[N-]S(=O)(=O...,2.40
3,CN1C=C[N+](C)=C1.FC(F)(F)S(=O)(=O)[N-]S(=O)(=O...,2.60
4,C1CNCCC1F.Cl,1.73


In [3]:
def is_valid_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        return mol is not None
    except:
        return False

valid_mask = df_clean['smiles'].apply(is_valid_smiles)
invalid = df_clean[~valid_mask]
if len(invalid) > 0:
    print(f"⚠️  无效 SMILES {len(invalid)} 条，已剔除：")
    print(invalid['smiles'].tolist())

df_clean = df_clean[valid_mask].reset_index(drop=True)
print(f"RDKit 验证后: {len(df_clean)} 行")

⚠️  无效 SMILES 6 条，已剔除：
['[CH-]1C=CC(=C1)P(C2=CC=CC=C2)C3=CC=CC=C3.', '[BF4-].CCCCCCCC[NH3+]', '[[O-]C(=O)C(F)(F)F].CCCCCCCC[NH3+]', '[CC1=CC=C(C=C1)S(R)(=O)=O].CCCCCCCC[NH3+]', '[I-].[I-].[NH3+]CCC[NH5+]', 'C1=C(C=CC2=C3N4C(=C12)N=[C]1=C2C=CC(=CC2=C2N1[P]41([N]4=C(C5=C(C=C(C=C5)OC5=C(C(CCC)(C)C)C=C(C(=C5)C(C)(C)CCC)OC)C4=N3)C3C4=C(C=CC(OC5=C(C=C(OC)C(=C5)C(CCC)(C)C)C(CCC)(C)C)=C4)C(=N2)N13)(OC)OC)OC1=C(C(CCC)(C)C)C=C(OC)C(=C1)C(C)(C)CCC)OC1=C(C(CCC)(C)C)C=C(C(=C1)C(C)(C)CCC)OC.CC']
RDKit 验证后: 243 行


[16:34:06] SMILES Parse Error: syntax error while parsing: [CH-]1C=CC(=C1)P(C2=CC=CC=C2)C3=CC=CC=C3.
[16:34:06] SMILES Parse Error: Failed parsing SMILES '[CH-]1C=CC(=C1)P(C2=CC=CC=C2)C3=CC=CC=C3.' for input: '[CH-]1C=CC(=C1)P(C2=CC=CC=C2)C3=CC=CC=C3.'
[16:34:06] SMILES Parse Error: syntax error while parsing: [BF4-].CCCCCCCC[NH3+]
[16:34:06] SMILES Parse Error: Failed parsing SMILES '[BF4-].CCCCCCCC[NH3+]' for input: '[BF4-].CCCCCCCC[NH3+]'
[16:34:06] SMILES Parse Error: syntax error while parsing: [[O-]C(=O)C(F)(F)F].CCCCCCCC[NH3+]
[16:34:06] SMILES Parse Error: Failed parsing SMILES '[[O-]C(=O)C(F)(F)F].CCCCCCCC[NH3+]' for input: '[[O-]C(=O)C(F)(F)F].CCCCCCCC[NH3+]'
[16:34:06] SMILES Parse Error: syntax error while parsing: [CC1=CC=C(C=C1)S(R)(=O)=O].CCCCCCCC[NH3+]
[16:34:06] SMILES Parse Error: Failed parsing SMILES '[CC1=CC=C(C=C1)S(R)(=O)=O].CCCCCCCC[NH3+]' for input: '[CC1=CC=C(C=C1)S(R)(=O)=O].CCCCCCCC[NH3+]'
[16:34:06] Explicit valence for atom # 6 N, 5, is greater than permit

In [4]:
# train / (val + test) 划分
df_train, df_temp = train_test_split(
    df_clean,
    test_size=VAL_RATIO + TEST_RATIO,
    random_state=SEED
)

# val / test 划分
val_size_adjusted = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
df_val, df_test = train_test_split(
    df_temp,
    test_size=val_size_adjusted,
    random_state=SEED
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# 保存
df_train.to_csv(f"{OUTPUT_DIR}/train.csv", index=False)
df_val.to_csv(f"{OUTPUT_DIR}/val.csv",     index=False)
df_test.to_csv(f"{OUTPUT_DIR}/test.csv",   index=False)

print(f"✅ train: {len(df_train)} | val: {len(df_val)} | test: {len(df_test)}")
print(f"已保存至: {OUTPUT_DIR}/")

✅ train: 194 | val: 24 | test: 25
已保存至: finetune_data/perovskite/


In [6]:
import subprocess

PKL_OUTPUT_DIR = OUTPUT_DIR.rstrip('/') + '_pkl'

cmd = [
    "python", "../../../../data_create/create_dataset.py",
    "--dataset_name", "gen",
    "--train_path", f"{OUTPUT_DIR}/train.csv",
    "--val_path",   f"{OUTPUT_DIR}/val.csv",
    "--test_path",  f"{OUTPUT_DIR}/test.csv",
    "--output_dir", PKL_OUTPUT_DIR,
    "--target_name", "y",
    "--frag_type", "brics",
]

print("运行命令:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("❌ 错误:", result.stderr)
else:
    print(f"✅ pkl 数据已保存至: {PKL_OUTPUT_DIR}/")

运行命令: python ../../../../data_create/create_dataset.py --dataset_name gen --train_path finetune_data/perovskite/train.csv --val_path finetune_data/perovskite/val.csv --test_path finetune_data/perovskite/test.csv --output_dir finetune_data/perovskite_pkl --target_name y --frag_type brics


FileNotFoundError: [Errno 2] No such file or directory: 'python'